<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c35-header.png" alt="Nextia Learning · Forecasting and Time Series: Predict What Comes Next" width="100%">

# Solution: From forecast to order and roster

**[From forecast to order and roster](https://learning.nextia-ai.com/courses/forecasting/m07/from-forecast-to-order-and-roster/)** · Forecasting and Time Series: Predict What Comes Next · Module 7, lesson 3 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** choose the order quantile from the cost of a lost sale and of a left-over item (the newsvendor idea), compute the expected cost of an order, compare the cost of ordering the forecast with ordering the best quantile, and staff the help desk to a quantile of the ticket forecast.

| | |
|---|---|
| **Time** | About 45 minutes, of which about 1 minute of waiting for the backtests |
| **Needs** | An internet connection for the setup cells. No account and no key. |
| **You should know** | Ranges, quantiles and coverage, from [A forecast is a range](https://learning.nextia-ai.com/courses/forecasting/m07/a-forecast-is-a-range/) and [Score the range](https://learning.nextia-ai.com/courses/forecasting/m07/score-the-range/). |
| **Tested** | Python 3.12 with pandas 2.3.3, statsforecast 2.1.1, mlforecast 1.1.0, hierarchicalforecast 1.5.3, LightGBM 4.7.0 and matplotlib 3.11.2, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/forecasting/m07/from-forecast-to-order-and-roster/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C35/M07-L03-from-forecast-to-order-and-roster/from-forecast-to-order-and-roster-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 2.3.3, statsforecast 2.1.1, and matplotlib for the charts. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==2.3.3 statsforecast==2.1.1 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
import pandas; print("pandas", pandas.__version__)
import statsforecast; print("statsforecast", statsforecast.__version__)

## Setup: get the practice data

Run the next cell. It downloads the Larkfield data of this course (about 2 MB) into a folder `larkfield-forecasting/data`, and the course's backtest helper `backtest.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: 403758 rows of daily sales (200 products, 3 regions), from 2024-04-01 to 2026-03-29.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-forecasting` (in Colab: the **Files** panel on the left), then run the cell again.

The Larkfield data is synthetic: we made it for this course. It has no real people, products or prices.

In [ ]:
# Setup: download the course data into larkfield-forecasting/data/ and check it.
import gzip, hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C35/"
FILES = {
    "calendar.csv": "a5d1d5a3b640a983de72a30fca0e627a4ef23e797b1d80619e9ffcaadf118844",
    "demand.csv.gz": "8b39303985cbdb9bb1f3c705cd927ed6aaeb6f9665bd56ad4c3f8cb2b8578c99",
    "orders.csv": "bd8059fd73fe294daa157f8a6b6ae3f58319fa644eaa7ff0d31122b02cf05e30",
    "price_changes.csv": "889b258be4e933b87065d81d0771d6cd9b630ee90223b0a6ab074129c42153cb",
    "products.csv": "198b0a3de6eea8128af68f3cbc3a5c6acd3a64ac854fff18d37660857be8cfbf",
    "promotions.csv": "ca147959faa4c36c757742de7121976ffe79d59b53049dbbc8337a7547541f4a",
    "roster_hourly.csv": "6c2e0e96881b36ded52ed19458515c71f025288ddaebd589b50c82a8ad7fa69b",
    "tickets_daily.csv": "36f7067db060fdd2bd084c408c0622370a012773849f243020336f1fc678b31d",
    "tickets_hourly.csv.gz": "43f7fbb3e360a8e4a0cb4e101ea6a6242a76f0c37204d76ee9413ea3adbedd92",
    "weather.csv": "43a199a8b235940e291195bfd026901b62a10189b29f3a3ad9527b5382544276",
    "backtest.py": "764ecd38c6e96632be6f0e139e05eadd7b07d14a4ece73d881fc7559c2fa203a",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-forecasting").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected):
    """Use a copy next to the notebook if there is one, or download it."""
    if path.exists() and sha256(path) == expected:
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-forecasting and run again.")

for name, expected in FILES.items():
    if name == "backtest.py":
        fetch(name, LABS + name, PROJECT / name, expected)
    else:
        fetch(name, LABS + "data/" + name, PROJECT / "data" / name, expected)

os.chdir(PROJECT)  # work in the project folder, as in the lesson
with gzip.open("data/demand.csv.gz", "rt") as f:
    header = f.readline()
    rows = [line.split(",", 3)[:3] for line in f]
print(f"Ready: {len(rows)} rows of daily sales ({len({r[2] for r in rows})} products, {len({r[1] for r in rows})} regions), "
      f"from {min(r[0] for r in rows)} to {max(r[0] for r in rows)}.")

## Setup: helpers

The next cell loads pandas as `pd`, numpy as `np` and the course's backtest helper as `bt`, and defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the backtest helper, and checks that do not show the answer.
import hashlib, json, logging, os, warnings
os.environ["NIXTLA_ID_AS_COL"] = "1"
import numpy as np
import pandas as pd
import backtest as bt
pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
warnings.filterwarnings("ignore")
logging.getLogger("lightning").setLevel(logging.ERROR)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

def load_sales(products=None):
    """Daily sales per product, the three regions added up, in the long format of the backtest:
    unique_id (the product), ds (the day), y (units sold), price, on_promotion, stock_out.
    The missing day 2025-10-19 gets the mean of the same weekday one week before and after."""
    d = pd.read_csv("data/demand.csv.gz", parse_dates=["date"])
    if products is not None:
        d = d[d["product_id"].isin(products)]
    g = d.groupby(["product_id", "date"], as_index=False).agg(
        y=("sales", "sum"), price=("price", "mean"), on_promotion=("on_promotion", "max"), stock_out=("stock_out", "max"))
    out = []
    for uid, s in g.rename(columns={"product_id": "unique_id", "date": "ds"}).groupby("unique_id"):
        s = s.set_index("ds").reindex(pd.date_range(s["ds"].min(), s["ds"].max(), freq="D"))
        gap = s["y"].isna()
        before, after = s["y"].shift(7), s["y"].shift(-7)
        s.loc[gap, "y"] = ((before + after) / 2).fillna(before).fillna(after)[gap].round()
        s = s.ffill()
        s["unique_id"] = uid
        out.append(s.rename_axis("ds").reset_index())
    return pd.concat(out, ignore_index=True)[["unique_id", "ds", "y", "price", "on_promotion", "stock_out"]]

# The course backtest (the same in every module): 13 origins, 28 days apart, the last one 2026-03-15.
ORIGINS = bt.origins("2026-03-15", n=13, every="28D")
BACKTEST_PRODUCTS = ['P001', 'P002', 'P005', 'P006', 'P009', 'P013', 'P018', 'P024', 'P025', 'P029', 'P032', 'P039', 'P045', 'P057', 'P068', 'P075', 'P077', 'P078', 'P080', 'P081', 'P087', 'P088', 'P093', 'P095', 'P108', 'P110', 'P121', 'P130', 'P133', 'P136', 'P137', 'P141', 'P145', 'P147', 'P148', 'P156', 'P158', 'P159', 'P166', 'P169', 'P172', 'P174', 'P176', 'P183', 'P184', 'P186', 'P191', 'P192', 'P194', 'P199']
PRICES = pd.read_csv("data/products.csv").set_index("product_id")["base_price"].to_dict()

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The best quantile for one product

Kenji orders parasols for the next 14 days. The ETS forecast says 92.4 parasols, with a standard deviation of 20.4: demand is a bell-shaped range around 92.4. A parasol costs 79.99 dollars. A lost sale costs 35% of the price (the margin). A left-over parasol costs 25% of the price (it is sold at a discount at the end of the season). These two costs are the course's assumptions.

For a demand of this shape, the expected units short and left over have a formula. It is the same formula as in the course's forecasting job (`plan._expected_short_over`, Module 8):

> **Predict.** Which order is cheaper on average: 93 parasols (the middle of the range), 97 or 119 (the 90% quantile)? Then run the cell.

In [ ]:
from scipy.stats import norm

MEAN, SD, PRICE = 92.4, 20.4, 79.99
LOST, LEFT = 0.35, 0.25          # shares of the price (assumptions)

def expected_short_over(order, mean, sd):
    """For a normal demand: the expected units short and the expected units left over."""
    z = (order - mean) / sd
    short = sd * (norm.pdf(z) - z * (1 - norm.cdf(z)))
    return short, (order - mean) + short

def expected_cost(order):
    short, over = expected_short_over(order, MEAN, SD)
    return (short * LOST + over * LEFT) * PRICE

for order in [93, 97, 119]:
    short, over = expected_short_over(order, MEAN, SD)
    print(f"order {order}: short {short:.1f}, left over {over:.1f}, expected cost {expected_cost(order):.0f} dollars, "
          f"stock-out chance {1 - norm.cdf((order - MEAN) / SD):.0%}")

> **Check.** You should see:
>
> ```text
> order 93: short 7.8, left over 8.4, expected cost 388 dollars, stock-out chance 49%
> order 97: short 6.0, left over 10.6, expected cost 382 dollars, stock-out chance 41%
> order 119: short 0.9, left over 27.5, expected cost 576 dollars, stock-out chance 10%
> ```

The order of 97 is the cheapest of the three. Each extra parasol is worth ordering while the chance of selling it, times the 28 dollars of a lost sale, is larger than the chance of not selling it, times the 20 dollars of a left-over item. That balance falls at the **critical ratio**: lost ÷ (lost + left) = 0.35 ÷ 0.60 = 0.583. So the best order is the **58% quantile** of demand. This is the **newsvendor** idea.

The next cell draws the expected cost of the order at every quantile, and marks the critical ratio.

In [ ]:
import matplotlib.pyplot as plt

ratio = LOST / (LOST + LEFT)
levels = np.arange(0.01, 1.0, 0.01)
orders = np.ceil(MEAN + norm.ppf(levels) * SD)
costs = [expected_cost(o) for o in orders]
best_order = int(np.ceil(MEAN + norm.ppf(ratio) * SD))
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(levels, costs, color="tab:blue")
ax.axvline(ratio, color="tab:green", linestyle="--", label=f"critical ratio {ratio:.3f}: order {best_order}")
ax.set_xlabel("Quantile Kenji orders at")
ax.set_ylabel("Expected cost (dollars)")
ax.legend()
plt.show()
print("best order:", best_order, " expected cost:", round(expected_cost(best_order)))

> **Check.** You should see a U-shaped curve, lowest near 58%, and `best order: 97  expected cost: 382`.

The bottom of the curve is flat. Between the 50% and the 65% quantile, the cost changes by only a few dollars. Far from the bottom, it rises fast: the 90% quantile costs 576 dollars, about 1.5 times as much.

> **Your turn.** Suppose a lost parasol cost 60% of the price instead of 35% (for example, a lost customer who does not come back). Compute the new critical ratio and put it in `ratio_60`, rounded to 3 digits. Then run the check cell.

In [ ]:
ratio_60 = round(0.60 / (0.60 + LEFT), 3)
print(ratio_60, "-> order", int(np.ceil(MEAN + norm.ppf(ratio_60) * SD)))

In [ ]:
expect('ratio_60', ratio_60, 0.706)

A lost sale that costs more moves the best order up: 0.706 is the 71% quantile, an order of 104 parasols.

## 2. Every product and every cut-off

Now the course backtest: ETS with its ranges, 13 cut-offs, 50 products. At each cut-off, Kenji orders for the next 14 days. The 14-day demand is the sum of 14 daily forecasts. We take its standard deviation from the daily 80% ranges, and we take the days as independent (as the forecasting job does). Then we compare two rules: order the forecast added up, or order the 58% quantile.

The cost of each order is computed with the **real** sales: a unit short costs 35% of the price, a unit left over 25%.

> **Predict.** Which rule costs less over the 13 cut-offs? Will the 58% rule run short more often or less often? Then run the cell. It takes about a minute.

In [ ]:
from statsforecast import StatsForecast
from statsforecast.models import AutoETS

def ranged(make_model, name, freq="D"):
    """A forecaster for the backtest: the model's forecast and its 80% and 90% ranges as quantile columns."""
    def forecaster(train, h, future):
        f = StatsForecast(models=[make_model()], freq=freq).forecast(
            df=train[["unique_id", "ds", "y"]], h=h, level=[80, 90])
        out = pd.DataFrame({"unique_id": f["unique_id"], "ds": f["ds"], "yhat": f[name],
                            "q5": f[f"{name}-lo-90"], "q10": f[f"{name}-lo-80"], "q50": f[name],
                            "q90": f[f"{name}-hi-80"], "q95": f[f"{name}-hi-90"]})
        return out.assign(**{c: out[c].clip(lower=0) for c in ["yhat", "q5", "q10", "q50", "q90", "q95"]})
    return forecaster

sales = load_sales(BACKTEST_PRODUCTS)
fc_ets = bt.backtest(sales, ranged(lambda: AutoETS(season_length=7), "AutoETS"), h=14, origins=ORIGINS)

def orders_at(fc, quantile=None):
    """One order per product and cut-off: the forecast added up (quantile=None) or a quantile of the 14-day demand."""
    daily_sd = (fc["q90"] - fc["q10"]) / (2 * norm.ppf(0.9))
    g = fc.assign(var=daily_sd ** 2).groupby(["unique_id", "origin"]).agg(
        sold=("y", "sum"), mean=("yhat", "sum"), var=("var", "sum")).reset_index()
    order = g["mean"] if quantile is None else g["mean"] + norm.ppf(quantile) * np.sqrt(g["var"])
    g["order"] = np.ceil(order.clip(lower=0))
    return g

def cost_per_cycle(g):
    """The real cost of the orders, per cut-off (one order cycle of 14 days for all 50 products)."""
    price = g["unique_id"].map(PRICES)
    short = (g["sold"] - g["order"]).clip(lower=0)
    over = (g["order"] - g["sold"]).clip(lower=0)
    return round(float((short * LOST * price + over * LEFT * price).sum() / g["origin"].nunique()))

for rule, q in [("forecast added up", None), ("the 58% quantile", ratio)]:
    g = orders_at(fc_ets, q)
    print(f"{rule:18} cost per order cycle: {cost_per_cycle(g)} dollars, "
          f"orders too small: {(g['sold'] > g['order']).mean():.1%}")

> **Check.** You should see `forecast added up  cost per order cycle: 16578 dollars, orders too small: 50.0%` and `the 58% quantile   cost per order cycle: 16294 dollars, orders too small: 44.0%`.

The same forecast, read differently, saves 284 dollars per order cycle. The 58% rule orders a little more, so it runs short less often. The saving is small because Larkfield's two costs are close (35% and 25%). With costs further apart, the best quantile moves further from the middle, and the saving grows.

> **Your turn.** Compute the cost per order cycle when Kenji orders the **90% quantile** "to be safe". Put it in `cost_90`. Then run the check cell.

In [ ]:
g90 = orders_at(fc_ets, 0.9)
cost_90 = cost_per_cycle(g90)
print(cost_90, f"orders too small: {(g90['sold'] > g90['order']).mean():.1%}")

In [ ]:
expect('cost_90', cost_90, 18009)

Ordering "to be safe" runs short only 20.3% of the time, and it costs 18009 dollars per cycle: 1715 dollars more than the 58% rule. Safety has a price, and here the price is larger than the lost sales it saves.

## 3. The help-desk roster

Grace plans agents for every open hour (07:00 to 21:59). One agent answers about 6 tickets an hour. An hour with one agent too few costs about 72 dollars (6 tickets that wait about a day longer). An extra agent hour costs 28 dollars. Both are the course's assumptions. The critical ratio is 72 ÷ (72 + 28) = 0.72: Grace staffs to the **72% quantile** of the tickets.

The next cell runs MSTL (day and week seasons, from the earlier module) on the hourly tickets at the same 13 cut-offs, one week ahead, with the last 12 weeks of history. Then it computes the cost of a week of roster at the point forecast and at the 72% quantile.

> **Predict.** Will the 72% roster cost more or less than the point roster? Then run the cell. It takes a few seconds.

In [ ]:
from statsforecast.models import MSTL

h = pd.read_csv("data/tickets_hourly.csv.gz", parse_dates=["timestamp"])
t = h.groupby("timestamp", as_index=False)["tickets"].sum()
tickets = pd.DataFrame({"unique_id": "tickets", "ds": t["timestamp"], "y": t["tickets"].astype(float)})
HOURLY_ORIGINS = [o + pd.Timedelta(hours=23) for o in ORIGINS]   # 23:00 of each cut-off day

mstl = ranged(lambda: MSTL(season_length=[24, 168], trend_forecaster=AutoETS(model="ZZN")), "MSTL", freq="h")
fc_t = bt.backtest(tickets, lambda train, h, future: mstl(train.tail(2016), h, future), h=168, origins=HOURLY_ORIGINS)

hourly_sd = (fc_t["q90"] - fc_t["q10"]) / (2 * norm.ppf(0.9))
fc_t["plan_72"] = (fc_t["yhat"] + norm.ppf(0.72) * hourly_sd).clip(lower=0)
roster = bt.roster_cost()   # agents = tickets / 6 rounded up; 28 dollars an extra hour, 72 a short hour
print("point forecast:  ", round(roster(fc_t, "yhat")), "dollars a week")
print("the 72% quantile:", round(roster(fc_t, "plan_72")), "dollars a week")

> **Check.** You should see `point forecast:   4044 dollars a week` and `the 72% quantile: 3718 dollars a week`.

Staffing to the 72% quantile saves 326 dollars a week. A short hour costs more than two extra hours here, so a few more agents in the busy hours pay for themselves.

## 4. Failure case: a quantile from a range that is too wide

The quantile rule trusts the range. The next cell does the same with Theta, which had the **best** ticket WAPE of the earlier module. Its ranges are far too wide.

> **Predict.** Theta's point forecast is more accurate than MSTL's. Will its 72% roster be cheaper too? Then run the cell. It takes about 10 seconds.

In [ ]:
from statsforecast.models import AutoTheta

theta = ranged(lambda: AutoTheta(season_length=168), "AutoTheta", freq="h")
fc_th = bt.backtest(tickets, lambda train, h, future: theta(train.tail(2016), h, future), h=168, origins=HOURLY_ORIGINS)
theta_sd = (fc_th["q90"] - fc_th["q10"]) / (2 * norm.ppf(0.9))
fc_th["plan_72"] = (fc_th["yhat"] + norm.ppf(0.72) * theta_sd).clip(lower=0)
print("Theta coverage of the 80% range:", bt.coverage(fc_th, "q10", "q90"))
print("point forecast:  ", round(roster(fc_th, "yhat")), "dollars a week")
print("the 72% quantile:", round(roster(fc_th, "plan_72")), "dollars a week")

> **Check.** You should see `Theta coverage of the 80% range: 0.996`, then `3883` and `5845` dollars a week.

Theta's 80% range holds almost every real value: it is far too wide. Its 72% quantile is far above its forecast, so the roster has many extra agents, and it costs more than the point roster. **Score the range before you decide from it.** The method with the best point forecast is not always the method with the cheapest decision.

## Next

You can now turn a forecast range into an order and a roster, and compare methods by the cost of their decisions. The [lesson page](https://learning.nextia-ai.com/courses/forecasting/m07/from-forecast-to-order-and-roster/) has the explorer, the comparison of every method and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [The forecasting job](https://learning.nextia-ai.com/courses/forecasting/m08/the-forecasting-job/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/forecasting/m07/from-forecast-to-order-and-roster/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.